# SVCM — ITI-95 — Récupération ValueSet

**Profil** : IHE ITI SVCM
**Acteur testé** : SVCM-Terminology Consumer
**Transaction** : SVCM-ITI-95
**Référence Gazelle** : https://interop.esante.gouv.fr/tm/testing/testsDefinition/viewTestPage.seam?id=12206&cid=48518

Récupérer le ValueSet `JDV-J245-Civilite-CISIS` par URL canonique, puis par `fullUrl`.

## Configuration

In [ ]:
import requests
import json
import time
from urllib.parse import quote

FHIR_BASE = "https://smt.esante.gouv.fr/fhir"
API_KEY   = "ZiaoDxF8Je0CfBNzu..."   # Remplacer par la clé API complète

HTTP_RETRIES = 3
HTTP_BACKOFF = 2

HEADERS_JSON = {
    "Accept": "application/fhir+json",
    "User-Agent": "SVCM-Consumer-CAD-CDM",
}

def http_get(url, headers=None):
    if headers is None:
        headers = HEADERS_JSON
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            print(f"  → GET {url}")
            r = requests.get(url, headers=headers, allow_redirects=True)
            if r.status_code == 200:
                return r
            raise Exception(f"HTTP {r.status_code}: {r.text[:200]}")
        except Exception as e:
            print(f"  ✗ tentative {attempt}/{HTTP_RETRIES} : {e}")
            if attempt == HTTP_RETRIES:
                raise
            time.sleep(HTTP_BACKOFF ** attempt)

def print_keys(data, *keys):
    subset = {k: data.get(k) for k in keys if k in data}
    print(json.dumps(subset, indent=2, ensure_ascii=False))

print("Configuration OK — prêt à exécuter les étapes.")


---
## Étapes 0–30 — Récupération du ValueSet JDV-J245-Civilite-CISIS

**Requête** : `GET /fhir/ValueSet?url=https://mos.esante.gouv.fr/NOS/JDV_J245-Civilite-CISIS/FHIR/JDV-J245-Civilite-CISIS&_format=json`
**Objectif** : Localiser le ValueSet par URL canonique, extraire le `fullUrl`, puis le récupérer directement.

In [ ]:
# Étape 0  — Construction de la requête
# Étape 10 — TRANSACTION : GET ValueSet par URL canonique
JDV_J245_URL = "https://mos.esante.gouv.fr/NOS/JDV_J245-Civilite-CISIS/FHIR/JDV-J245-Civilite-CISIS"
url_search = f"{FHIR_BASE}/ValueSet?url={quote(JDV_J245_URL)}&_format=json"

r_search = http_get(url_search)
bundle = r_search.json()

# Étape 20 — Réception et intégration
entries = bundle.get("entry", [])
assert len(entries) > 0, f"Aucune entrée trouvée pour {JDV_J245_URL}"

vs_entry   = entries[0]
full_url   = vs_entry.get("fullUrl", "")
vs_summary = vs_entry.get("resource", {})

print(f"Entrées trouvées : {len(entries)}")
print(f"fullUrl          : {full_url}")
print(f"  name    : {vs_summary.get('name')}")
print(f"  title   : {vs_summary.get('title')}")
print(f"  version : {vs_summary.get('version')}")
print(f"  status  : {vs_summary.get('status')}")

# Étape 30 — PREUVE : GET par fullUrl, affichage des champs clés
print(f"\n[PREUVE étape 30] GET par fullUrl : {full_url}")
r_direct = http_get(f"{full_url}?_format=json")
vs = r_direct.json()

print_keys(vs, "resourceType", "id", "url", "version", "name", "title", "status")
compose = vs.get("compose", {})
includes = compose.get("include", [])
total_concepts = sum(len(inc.get("concept", [])) for inc in includes)
print(f"\nNombre de concepts : {total_concepts}")
if total_concepts <= 20:
    for inc in includes:
        for c in inc.get("concept", []):
            print(f"  {c.get('code')} — {c.get('display')}")
